In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
def clean_generic_name(name):
    if pd.isna(name) or name == 'nan':
        return name
    
    # 1. 統一標點符號：全型斜線與逗號轉半型，移除常見括號內容
    name = name.replace('／', '/').replace('，', ',')
    name = re.sub(r'[\(\uff08].*?[\)\uff09]', '', name)
    
    # 2. 強力移除劑量與單位 (處理 100,000, 1%, 80 mg/ml, 2.4 MIU 等)
    # 匹配數字、逗號、點、百分比，後接常見容量單位
    name = re.sub(r'[\d\.,\s]+(g|mg|ml|units|miu|％|%)\b', ' ', name, flags=re.IGNORECASE)
    # 針對單獨殘留的 /ml 或 %
    name = re.sub(r'/\s*ml|%', '', name, flags=re.IGNORECASE)
    
    # 3. 移除特定劑型關鍵字與無義詞
    forms = ['inj', 'tab', 'cap', 'susp', 'iv', 'oral', 'for', 'solution', 'hydrate', 'sodium', 'benzathine']
    pattern_forms = r'\b(' + '|'.join(forms) + r')\b'
    name = re.sub(pattern_forms, '', name, flags=re.IGNORECASE)

    # 4. 特定藥名結構處理 (依據您的要求)
    # Amphotericin B liposome -> Amphotericin B/liposome
    name = re.sub(r'Amphotericin B liposome', 'Amphotericin B/liposome', name, flags=re.IGNORECASE)

    # 5. 【核心強化】消除斜線 (/) 前後的任何空格
    # \s* 代表 0 到多個空白字元
    name = re.sub(r'\s*/\s*', '/', name)

    
    # 5. 清理殘留符號：移除多餘空格、末尾點號與斜線
    name = re.sub(r'\s+', ' ', name) # 多空格轉單空格
    name = name.strip(' ./,')       # 移除前後的空格、點、斜線、逗號
    
    # 6. 字典對照 (處理特殊轉換)
    mapping = {
        'R +I': 'ifampin/Isoniazid',
        'R 300 +I 150': 'ifampin/Isoniazid',
        'Penicillin G .': 'Penicillin G',
        'Penicillin G benzathine': 'Penicillin G',
        'Baktar': 'Sulfamethoxazole/Trimethoprim',
        'Clindamycin 1 ％' : 'Clindamycin',
        'Penicillin 5 MU' : 'Penicillin',
        'Rifampin, Isoniazid and Ethambutol' : 'Rifampin/Isoniazid/Ethambutol',
        'Minocycline injection' : 'Minocycline',
        'Amoxicillin/Clavulanate': 'Amoxicillin/Clavulanic acid'
    }
    
    # 如果完全符合字典 key，或是處理後變成 key 的樣子就轉換
    return mapping.get(name, name)

In [3]:
df2024 = pd.read_csv(r'C:\Users\482525\Desktop\敗血症資料\2024\2024Sepsis00114.csv', encoding='big5', dtype={'VERIFYDATE': str, 'STARTTIME': str, 'ENDTIME': str, 0: str, 21: str, 22: str})
df2025 = pd.read_csv(r'C:\Users\482525\Desktop\敗血症資料\2025\2025Sepsis00114.csv', encoding='big5', dtype={'VERIFYDATE': str, 'STARTTIME': str, 'ENDTIME': str, 0: str, 21: str, 22: str})

df2024['VERIFYDATE'] = pd.to_datetime(df2024['VERIFYDATE'], format='%Y%m%d', errors='coerce')
df2025['VERIFYDATE'] = pd.to_datetime(df2025['VERIFYDATE'], format='%Y%m%d', errors='coerce')
df2024['STARTTIME'] = pd.to_datetime(df2024['STARTTIME'], format='%Y%m%d%H%M%S', errors='coerce')
df2024['ENDTIME'] = pd.to_datetime(df2024['ENDTIME'], format='%Y%m%d%H%M%S', errors='coerce')
df2025['STARTTIME'] = pd.to_datetime(df2025['STARTTIME'], format='%Y%m%d%H%M%S', errors='coerce')
df2025['ENDTIME'] = pd.to_datetime(df2025['ENDTIME'], format='%Y%m%d%H%M%S', errors='coerce')

table14 = pd.concat([df2024, df2025], ignore_index=True)

In [4]:
table14 = table14.dropna(how='all')
table14 = table14[table14['ACCOUNTNO'].notna()]
table14 = table14.loc[:, ~table14.columns.str.contains('^Unnamed')]

In [5]:
len(table14), len(table14['ACCOUNTNO'].unique())

(86059, 27968)

In [6]:
table14['VERIFYDATE'] = pd.to_datetime(table14['VERIFYDATE'], format='%Y%m%d%H%M', errors='coerce')
table14['STARTTIME'] = pd.to_datetime(table14['STARTTIME'], format='%Y%m%d%H%M', errors='coerce')
table14['ENDTIME'] = pd.to_datetime(table14['ENDTIME'], format='%Y%m%d%H%M', errors='coerce')

In [7]:
# table14[table14['ACCOUNTNO'] == 'I11300000002']

In [8]:
table14['GENERICNAME_Clear'] = table14['GENERICNAME'].apply(clean_generic_name)

In [9]:
table14['GENERICNAME_Clear'].unique()

array(['Flomoxef', 'Tenofovir alafenamide', 'Cefixime',
       'Amoxicillin/Clavulanic acid', 'Metronidazole', 'Cefazolin',
       'Ceftriaxone', 'Cefoperazone/sulbactam', 'Gentamicin',
       'Cefadroxil', 'Oseltamivir', 'Baloxavir marboxil', 'Clindamycin',
       'Piperacillin/Tazobactam', 'Cefepime', 'Azithromycin',
       'Levofloxacin', 'Acyclovir', 'Cefuroxime', 'Ciprofloxacin',
       'Doxycycline', 'Peramivir', 'Vancomycin', 'Amoxicillin',
       'Valaciclovir', 'Nystatin', 'Ceftazidime', 'Penicillin',
       'Doripenem', 'Ertapenem', 'Cefotaxime', 'Cephalexin', 'Oxacillin',
       'Sulfamethoxazole/Trimethoprim', 'Ampicillin', 'Itraconazole',
       'Fluconazole', 'Amikacin', 'Moxifloxacin', 'Clarithromycin',
       'Anidulafungin', 'Ceftazidime/Avibactam', 'Fenticonazole',
       'Fosfomycin', 'Pipemidic acid', 'Cefoxitin',
       'Rifampin/Isoniazid/Ethambutol', 'Pyrazinamide', 'Teicoplanin',
       'Meropenem', 'Minocycline', 'ifampin/Isoniazid', 'Ceftizoxime',
       'Famc

In [10]:
table14['GENERICNAME_Clear'][table14['GENERICNAME_Clear'].isna() == True]

Series([], Name: GENERICNAME_Clear, dtype: object)

In [11]:
table14[['ACCOUNTNO', 'GENERICNAME_Clear']]

,ACCOUNTNO,GENERICNAME_Clear
0,I11300000002,Flomoxef
1,I11300000002,Flomoxef
2,I11300000002,Flomoxef
3,I11300000002,Tenofovir alafenamide
4,I11300000002,Tenofovir alafenamide
...,...,...
86055,I11400060720,Piperacillin/Tazobactam
86056,I11400060731,Flomoxef
86057,I11400060731,Flomoxef
86058,I11400060731,Flomoxef


In [12]:
infection_cols = [
    "INFECTIONSITE1",
    "INFECTIONSITE2",
    "INFECTIONSITE3",
    "INFECTIONSITE4",
    "INFECTIONSITE5",
    "INFECTIONSITE9"
]

for col in infection_cols:
    table14[col] = (table14[col] == "Y").astype(int)

In [13]:
table14['INFECTIONSITE1']

0        1
1        0
2        1
3        0
4        0
        ..
86055    1
86056    1
86057    0
86058    1
86059    1
Name: INFECTIONSITE1, Length: 86059, dtype: int32

In [14]:
BACTERIA_cols = [
    "BACTERIA1",
    "BACTERIA2",
    "BACTERIA3",
    "BACTERIA4",
    "BACTERIA5",
    "BACTERIA9"
]

for col in BACTERIA_cols:
    table14[col] = (table14[col] == "Y").astype(int)

In [15]:
table14['BACTERIA1']

0        0
1        0
2        1
3        0
4        0
        ..
86055    0
86056    0
86057    0
86058    0
86059    0
Name: BACTERIA1, Length: 86059, dtype: int32

In [16]:
table14['AUTIBIOTICRANK'].unique()

array(['A32', 'A11', 'A21', 'A42', 'A22', 'A52', 'C42'], dtype=object)

In [17]:
rank_mapping = {
    'A11': 1,
    'A21': 2,
    'A22': 2,
    'A32': 2,
    'A42': 3,
    'A52': 3,
    'C42': 3
}

In [18]:
table14['AUTIBIOTIC_GROUP'] = table14['AUTIBIOTICRANK'].map(rank_mapping)

In [19]:
# last_index = table14.groupby('ACCOUNTNO')['VERIFYDATE'].min().reset_index()
# table14_last = pd.merge(table14, last_index, on=['ACCOUNTNO', 'VERIFYDATE'])

In [20]:
first_time = table14.groupby('ACCOUNTNO')['VERIFYDATE'].transform('min')

# day 3 (起)
start_time = first_time + pd.Timedelta(days=2)

# day 7 (終)
end_time = first_time + pd.Timedelta(days=7)

In [21]:
table14_D3_D7 = table14[(table14['VERIFYDATE'] >= start_time) & (table14['VERIFYDATE'] < end_time)]

In [22]:
table14_D3_D7 = table14_D3_D7[['ACCOUNTNO', 'GENERICNAME_Clear']].drop_duplicates()

In [23]:
table14_D3_D7.head(10)

,ACCOUNTNO,GENERICNAME_Clear
2,I11300000002,Flomoxef
3,I11300000002,Tenofovir alafenamide
5,I11300000002,Cefixime
16,I11300000015,Ceftriaxone
37,I11300000038,Clindamycin
49,I11300000040,Cefepime
51,I11300000040,Azithromycin
64,I11300000103,Cefuroxime
65,I11300000103,Cefixime
67,I11300000105,Ciprofloxacin


In [24]:
table14_D3_D7['GENERICNAME_Clear'].unique(), len(table14_D3_D7['GENERICNAME_Clear'].unique())

(array(['Flomoxef', 'Tenofovir alafenamide', 'Cefixime', 'Ceftriaxone',
        'Clindamycin', 'Cefepime', 'Azithromycin', 'Cefuroxime',
        'Ciprofloxacin', 'Amoxicillin/Clavulanic acid',
        'Cefoperazone/sulbactam', 'Cefazolin', 'Ceftazidime', 'Cefotaxime',
        'Levofloxacin', 'Cefadroxil', 'Oxacillin',
        'Sulfamethoxazole/Trimethoprim', 'Piperacillin/Tazobactam',
        'Acyclovir', 'Cephalexin', 'Metronidazole', 'Doripenem',
        'Itraconazole', 'Fluconazole', 'Amikacin', 'Clarithromycin',
        'Anidulafungin', 'Ceftazidime/Avibactam', 'Baloxavir marboxil',
        'Gentamicin', 'Doxycycline', 'Penicillin', 'Ampicillin',
        'Nystatin', 'Amoxicillin', 'Fosfomycin', 'Minocycline',
        'Meropenem', 'Ceftizoxime', 'Fenticonazole', 'Vancomycin',
        'Erythromycin', 'Micafungin', 'Ampicillin/Sulbactam',
        'Pyrazinamide', 'Rifampin', 'Ethambutol', 'Isoniazid', 'Peramivir',
        'Griseofulvin', 'Nemonoxacin', 'Rifampin/Isoniazid/Ethambutol',


In [25]:
abx14 = (table14_D3_D7.assign(value=1)
                     .pivot_table(index=['ACCOUNTNO'],columns='GENERICNAME_Clear',values='value',fill_value=0))

In [26]:
# add = table14_last.groupby('ACCOUNTNO')['AUTIBIOTIC_GROUP'].max().reset_index()
# abx14 = abx14.merge(add, on='ACCOUNTNO', how='left')

In [27]:
# infection_cols = ['INFECTIONSITE1', 'INFECTIONSITE2', 'INFECTIONSITE3', 
#                   'INFECTIONSITE4', 'INFECTIONSITE5', 'INFECTIONSITE9']

# for col in infection_cols:
#     table14[col] = table14[col].map({'Y': 1, 'N': 0}).fillna(0).astype(int)

# binary OTHERINFECTIONSITE 
table14['OTHERINFECTIONSITE_flag'] = (
    table14['OTHERINFECTIONSITE'].fillna('').str.strip().ne('').astype(int)
)

infects_summary = table14.groupby('ACCOUNTNO').agg({
    **{col: 'max' for col in infection_cols}, 
    'OTHERINFECTIONSITE_flag': 'max'
}).reset_index()


abx14 = abx14.merge(infects_summary, on='ACCOUNTNO', how='left').fillna(0)

In [28]:
abx14.columns

Index(['ACCOUNTNO', 'Acyclovir', 'Amikacin', 'Amoxicillin',
       'Amoxicillin/Clavulanic acid', 'Amphotericin B/liposome', 'Ampicillin',
       'Ampicillin/Sulbactam', 'Anidulafungin', 'Azithromycin',
       'Baloxavir marboxil', 'Cefadroxil', 'Cefazolin', 'Cefepime', 'Cefixime',
       'Cefoperazone/sulbactam', 'Cefotaxime', 'Cefoxitin', 'Ceftazidime',
       'Ceftazidime/Avibactam', 'Ceftizoxime', 'Ceftriaxone', 'Cefuroxime',
       'Cephalexin', 'Ciprofloxacin', 'Clarithromycin', 'Clindamycin',
       'Colistin', 'Dicloxacillin', 'Doripenem', 'Doxycycline', 'Ertapenem',
       'Erythromycin', 'Ethambutol', 'Famciclovir', 'Fenticonazole',
       'Flomoxef', 'Fluconazole', 'Fosfomycin', 'Ganciclovir', 'Gentamicin',
       'Griseofulvin', 'Imipenem/Cilastatin', 'Isoniazid', 'Itraconazole',
       'Levofloxacin', 'Linezolid', 'Meropenem', 'Metronidazole', 'Micafungin',
       'Minocycline', 'Moxifloxacin', 'Nemonoxacin', 'Nystatin', 'Oseltamivir',
       'Oxacillin', 'Penicillin', 'Pe

In [29]:
abx14

,ACCOUNTNO,Acyclovir,Amikacin,Amoxicillin,Amoxicillin/Clavulanic acid,Amphotericin B/liposome,Ampicillin,Ampicillin/Sulbactam,Anidulafungin,Azithromycin,...,abacavir/lamivudine/dolutegravir,ifampin/Isoniazid,tenofovir/emtricitabine/bictegravir,INFECTIONSITE1,INFECTIONSITE2,INFECTIONSITE3,INFECTIONSITE4,INFECTIONSITE5,INFECTIONSITE9,OTHERINFECTIONSITE_flag
0,I11300000002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1,0,0,0,0,0,0
1,I11300000015,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0,0,1,1,0,0,0
2,I11300000038,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0,0,0,0,0,0,0
3,I11300000040,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1,1,0,0,0,0,0
4,I11300000103,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6168,I11400060661,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0,1,1,0,0,0,0
6169,I11400060667,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0,0,0,0,0,0,0
6170,I11400060687,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0,1,0,0,0,0,0
6171,I11400060720,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1,0,0,0,0,0,0


In [30]:
table14_clear = abx14.reset_index(drop=True)

In [31]:
# table14_clear.to_csv('table14_clear.csv', index=False)

In [32]:
table14_clear = table14_clear.drop(columns=['ACCOUNTNO']) # 'AUTIBIOTIC_GROUP'

In [33]:
start_index = table14_clear.columns.get_loc('Acyclovir')
end_index = table14_clear.columns.get_loc('tenofovir/emtricitabine/bictegravir')

abx_cols = table14_clear.columns[start_index:end_index+1]
col_sum = table14_clear[abx_cols].sum()

final_cols = col_sum[col_sum >= 0].index.tolist() # 抗生素
base_cols = [c for c in table14_clear.columns if c not in abx_cols] # 感染部位
data_filter = table14_clear[base_cols + final_cols]

feature_cols = list(set(data_filter.columns) - set(abx_cols))
X = data_filter[feature_cols]
y = data_filter[final_cols]

X.shape, X.sum(), y.shape, y.sum()

((6173, 7),
 OTHERINFECTIONSITE_flag     314
 INFECTIONSITE4              179
 INFECTIONSITE3             1782
 INFECTIONSITE9              352
 INFECTIONSITE5              741
 INFECTIONSITE1             3422
 INFECTIONSITE2             2029
 dtype: int64,
 (6173, 76),
 Acyclovir                               14.0
 Amikacin                                29.0
 Amoxicillin                             54.0
 Amoxicillin/Clavulanic acid            614.0
 Amphotericin B/liposome                  1.0
                                        ...  
 Vancomycin                             113.0
 Voriconazole                             1.0
 abacavir/lamivudine/dolutegravir         1.0
 ifampin/Isoniazid                        4.0
 tenofovir/emtricitabine/bictegravir     11.0
 Length: 76, dtype: float64)

In [34]:
abx14 = y

result = {}
for i in abx14.columns:
    summ = abx14[i].sum()
    if summ >= 0:
       result[i] = summ

for key, value in sorted(result.items(), key=lambda x: x[1], reverse=True):
    print(f'{key}: {value}')
    
print(len(result))

Flomoxef: 1102.0
Piperacillin/Tazobactam: 1038.0
Cefixime: 698.0
Amoxicillin/Clavulanic acid: 614.0
Cefuroxime: 551.0
Levofloxacin: 475.0
Ciprofloxacin: 452.0
Cefoperazone/sulbactam: 431.0
Metronidazole: 382.0
Azithromycin: 368.0
Cefazolin: 340.0
Clindamycin: 319.0
Fosfomycin: 229.0
Ceftriaxone: 207.0
Ceftazidime: 201.0
Tenofovir alafenamide: 185.0
Doxycycline: 176.0
Gentamicin: 118.0
Cefadroxil: 114.0
Vancomycin: 113.0
Cefotaxime: 94.0
Cephalexin: 94.0
Cefepime: 92.0
Fluconazole: 90.0
Nemonoxacin: 74.0
Nystatin: 70.0
Sulfamethoxazole/Trimethoprim: 69.0
Meropenem: 68.0
Ceftizoxime: 64.0
Amoxicillin: 54.0
Minocycline: 45.0
Penicillin: 35.0
Ampicillin: 33.0
Amikacin: 29.0
Baloxavir marboxil: 28.0
Doripenem: 21.0
Erythromycin: 18.0
Oxacillin: 18.0
Peramivir: 18.0
Clarithromycin: 17.0
Moxifloxacin: 17.0
Pyrazinamide: 17.0
Ampicillin/Sulbactam: 16.0
Ethambutol: 16.0
Acyclovir: 14.0
Teicoplanin: 14.0
Itraconazole: 13.0
Isoniazid: 12.0
Ertapenem: 11.0
tenofovir/emtricitabine/bictegravir: 11.0

In [35]:
col_sum = abx14.sum()

abx14_filter = abx14.loc[:, col_sum >= 0]

In [36]:
abx14_filter.columns

Index(['Acyclovir', 'Amikacin', 'Amoxicillin', 'Amoxicillin/Clavulanic acid',
       'Amphotericin B/liposome', 'Ampicillin', 'Ampicillin/Sulbactam',
       'Anidulafungin', 'Azithromycin', 'Baloxavir marboxil', 'Cefadroxil',
       'Cefazolin', 'Cefepime', 'Cefixime', 'Cefoperazone/sulbactam',
       'Cefotaxime', 'Cefoxitin', 'Ceftazidime', 'Ceftazidime/Avibactam',
       'Ceftizoxime', 'Ceftriaxone', 'Cefuroxime', 'Cephalexin',
       'Ciprofloxacin', 'Clarithromycin', 'Clindamycin', 'Colistin',
       'Dicloxacillin', 'Doripenem', 'Doxycycline', 'Ertapenem',
       'Erythromycin', 'Ethambutol', 'Famciclovir', 'Fenticonazole',
       'Flomoxef', 'Fluconazole', 'Fosfomycin', 'Ganciclovir', 'Gentamicin',
       'Griseofulvin', 'Imipenem/Cilastatin', 'Isoniazid', 'Itraconazole',
       'Levofloxacin', 'Linezolid', 'Meropenem', 'Metronidazole', 'Micafungin',
       'Minocycline', 'Moxifloxacin', 'Nemonoxacin', 'Nystatin', 'Oseltamivir',
       'Oxacillin', 'Penicillin', 'Peramivir', 'Pi

In [37]:
mask = abx14_filter.sum(axis=1) > 0
abx14_final = abx14_filter[mask].reset_index()

In [38]:
abx14_final

,index,Acyclovir,Amikacin,Amoxicillin,Amoxicillin/Clavulanic acid,Amphotericin B/liposome,Ampicillin,Ampicillin/Sulbactam,Anidulafungin,Azithromycin,...,Tenofovir,Tenofovir alafenamide,Terbinafine,Tetracycline,Valaciclovir,Vancomycin,Voriconazole,abacavir/lamivudine/dolutegravir,ifampin/Isoniazid,tenofovir/emtricitabine/bictegravir
0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6168,6168,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6169,6169,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6170,6170,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6171,6171,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
